# Qwen3-VL-8B on the repaired BEV raster (pre-registered, D-053)

A separate notebook so `stage_e_kaggle.ipynb` is never edited for this run. The cells below
are that notebook's, unchanged, except the parameter cell, which is pinned.

**Same data:** attach the existing `thesis-bundle` dataset (`outputs/colab_bundle.tar.gz`,
the 2026-09-22 upload). Nothing new to upload.

**Setup:** Settings -> Accelerator -> **GPU T4 x2**; Settings -> Internet -> **On**.
Then **Save Version -> Save & Run All (Commit)**. About 1.5 h (150 frames at ~36 s).

**Take home exactly one file:** `/kaggle/working/results/v3_bev_r2__Qwen3-VL-8B-Instruct.jsonl`.
The folder also holds copies of every completed run from the bundle (that is how resume
works); those are the old files and must not be downloaded as results.


In [ ]:
import os, sys, json, glob, pathlib, shutil, tarfile

# Kaggle AUTO-EXTRACTS uploaded archives, and the folder it extracts into is NOT
# predictable - it has been both <dataset>/colab_bundle/ and <dataset>/ directly. Find
# the bundle ROOT by its marker file rather than by name; a hardcoded path has broken
# twice already.
_marker = glob.glob('/kaggle/input/**/outputs/label_schema.json', recursive=True)
if _marker:
    WORK = pathlib.Path(_marker[0]).parent.parent      # already extracted: read in place
    print('bundle found at:', WORK)
else:
    _tars = glob.glob('/kaggle/input/**/*.tar.gz', recursive=True)
    if not _tars:
        print('NOTHING MATCHED. Actual layout under /kaggle/input:')
        for q in sorted(pathlib.Path('/kaggle/input').rglob('*'))[:40]:
            print('   ', q)
        raise SystemExit('no bundle found - send the listing above')
    WORK = pathlib.Path('/kaggle/working/work'); WORK.mkdir(parents=True, exist_ok=True)
    with tarfile.open(_tars[0]) as t:
        t.extractall(WORK)
    print('extracted', _tars[0], '->', WORK)

OUT = pathlib.Path('/kaggle/working/results'); OUT.mkdir(parents=True, exist_ok=True)

# /kaggle/input is READ-ONLY, so completed rows are copied into the writable dir before
# run_batch appends to them.
for f in (WORK/'results').glob('*.jsonl'):
    dst = OUT/f.name
    if not dst.exists():
        shutil.copy(f, dst)
        print(f'resuming from {f.name}: {sum(1 for _ in dst.open())} rows')

assert (WORK/'images').is_dir() and (WORK/'src').is_dir(), f'bundle looks wrong at {WORK}'
print('camera:', len(list((WORK/'images').glob('*.jpg'))),
      ' bev:', len(list((WORK/'bev').glob('*.png'))))


In [ ]:
!pip -q install -U "transformers>=4.49" accelerate bitsandbytes
import torch

# bitsandbytes 4-bit NF4 needs compute capability >= 7.5. Kaggle offers T4 x2 (7.5, fine)
# and P100 (6.0, NOT fine) from the same pool. Without 4-bit the 7B needs ~15 GB in fp16
# against the P100's 16 GB, leaving nothing for the KV cache - so this is a hard stop, not
# a slow path. Checked here rather than 20 minutes into a model download.
_cap = torch.cuda.get_device_capability(0)
_name = torch.cuda.get_device_name(0)
assert _cap >= (7, 5), (
    f'{_name} has compute capability {_cap[0]}.{_cap[1]}; bitsandbytes 4-bit needs 7.5+. '
    'Switch Settings -> Accelerator to GPU T4 x2 (Turing, 7.5). P100 is Pascal 6.0 and '
    'cannot run this setup at all.')
print(f'{_name}  cc {_cap[0]}.{_cap[1]}  n={torch.cuda.device_count()}  '
      f'{torch.cuda.get_device_properties(0).total_memory/2**30:.0f} GB')


## Parameters: pinned to this run


In [ ]:
# PINNED. This notebook exists for one run: the repaired raster BEV arm on Qwen3-VL-8B,
# over E10's 150-frame slice (tokens[:150]), so it pairs frame for frame with E10's Qwen3
# camera run. Do not edit; any other run belongs in stage_e_kaggle.ipynb.
CONDITION = 'v3_bev_r2'
MODEL     = 'Qwen/Qwen3-VL-8B-Instruct'
SLICE     = 150
LIMIT     = None                            # e.g. 3 for a smoke test


In [ ]:
sys.path.insert(0, str(WORK))
import src.vlm as V, src.prompts as P

# F5 asks an EXTERNAL benchmark's questions, so the unit of work is a QUESTION, not a frame:
# 1,471 questions over 111 frames, keyed '<sample_token>:<index>'. Batching a frame's ~13
# questions into one call is ~13x cheaper and was rejected on measurement -- 61.0% of the
# questions contain another question's answer verbatim in their own text, so a batched prompt
# hands the model a candidate list read off its own input and the number stops being
# comparable to the published ones.
F5 = CONDITION == 'f5_nuscenes_qa'
if F5:
    _spec = json.loads((WORK/'outputs/f5_questions.json').read_text())
    F5Q = {q['qid']: q for q in _spec['questions']}
    print(f"F5: {_spec['n_questions']} questions over {_spec['n_frames']} val-split frames")

# F-065 guard. preflight.py checks the prompt on the laptop; it cannot see what Kaggle
# actually loaded. A stale bundle, or a module cached from an earlier interactive run,
# would silently reinstate the prompt that told the model to think and then forbade it -
# 5.6 h producing a number that looked like a finding.
if F5:
    import src.nuqa as Q
    _p = Q.ANSWER_PROMPT     # a per-question prompt, not one of the tag prompts
else:
    _p = P.build_prompt(CONDITION, **({'n_frames': 5} if CONDITION == 'v6_temporal' else {}))
_asks = 'think step by step' in _p or 'reasoning first' in _p
assert not (_asks and 'nothing else' in _p), (
    f'{CONDITION} both requests reasoning and forbids it. This is the pre-F-065 prompt, '
    'so src/prompts.py in the bundle is STALE. Re-upload the bundle, then Restart session.')
print(f'prompt check: {CONDITION} ok ({len(_p)} chars)')

# HARD FAIL, not a warning. The Colab run silently fell back to the full 1,800-frame list
# and spent ten hours more than intended before anyone noticed. A run this expensive must
# refuse to start on the wrong subset rather than print a line nobody reads.
_vlm = WORK/'outputs/vlm_subset_tokens.json'
assert _vlm.exists(), (
    'vlm_subset_tokens.json is missing from the bundle. Rebuild it with '
    'src.data.build_colab_bundle() - running the full 1,800 costs 14.6 h per condition '
    'instead of 5.2 h.')
_s = json.loads(_vlm.read_text())
tokens = _s['sample_tokens']
schema = json.loads((WORK/'outputs/label_schema.json').read_text())
bev_text = {json.loads(l)['sample_token']: json.loads(l)['description']
            for l in (WORK/'outputs/bev_symbolic.jsonl').open()}
# The REPAIRED text arm reads its own file. The published one stays exactly as it was, so
# the two runs are comparable and the re-run cannot silently score the old description
# (F-064b). Only loaded when the repaired arm is the one being run.
BEV_R2 = ('v3_bev_r2', 'v3b_bev_symbolic_r2')
if CONDITION in BEV_R2:
    _r2 = WORK/'outputs/bev_symbolic_r2.jsonl'
    assert _r2.exists(), ('the bundle has no bev_symbolic_r2.jsonl: it predates the BEV '
                          'repair. Re-upload it, or this run scores the old input.')
    bev_text_r2 = {json.loads(l)['sample_token']: json.loads(l)['description']
                   for l in _r2.open()}

assert len(tokens) == _s['n_frames'] == 628, f'unexpected subset size {len(tokens)}'
print(f"execution subset: {len(tokens)} frames, min positives {_s['min_positives']} "
      f"({_s['min_positives_tag']}), ~{_s['est_hours_per_condition_at_29s']} h per condition")

# E9. The 5-frame windows are precomputed on the laptop and travel in the bundle:
# rebuilding them here would need the 440 MB trainval metadata, which is not shipped.
_seqp = WORK/'outputs/temporal_sequences.json'
if CONDITION == 'v6_temporal':
    assert _seqp.exists(), (
        'temporal_sequences.json missing. Rebuild the bundle with '
        'src.data.build_colab_bundle(temporal=True) - without it the prompt promises five '
        'frames and the model gets one, which is the original query_multiframe defect.')
    SEQS = json.loads(_seqp.read_text())['sequences']
    _missing = [t for t in tokens for f in SEQS[t] if not (WORK/f'images/{f}.jpg').exists()]
    assert not _missing, f'{len(_missing)} sequence frames absent from the bundle'
    print(f'temporal: {len(SEQS)} sequences, '
          f'{len({f for t in tokens for f in SEQS[t]})} distinct frames')

def images_for(tok):
    if F5:                       # tok is a question id; the image is its frame's
        return [WORK/f"images/{F5Q[tok]['sample_token']}.jpg"]
    cam, bev = WORK/f'images/{tok}.jpg', WORK/f'bev/{tok}.png'
    if CONDITION == 'v6_temporal':
        return [WORK/f'images/{f}.jpg' for f in SEQS[tok]]
    bev_r2 = WORK/f'bev_r2/{tok}.png'
    return {'v1_structured': [cam], 'v2_cot': [cam], 'v3_bev': [bev],
            'v3b_bev_symbolic': [], 'v4_both': [cam, bev],
            'v3_bev_r2': [bev_r2], 'v3b_bev_symbolic_r2': []}[CONDITION]

if CONDITION == 'v3b_bev_symbolic':
    text_for = lambda t: bev_text[t]
elif CONDITION == 'v3b_bev_symbolic_r2':
    text_for = lambda t: bev_text_r2[t]
else:
    text_for = None

# HARD FAIL on a missing repaired raster, for the reason F-048 exists: a hole does not
# stop the run, it shortens it, and a shorter results file scores as a worse model.
if CONDITION == 'v3_bev_r2':
    _gap = [t for t in tokens if not (WORK/f'bev_r2/{t}.png').exists()]
    assert not _gap, f'{len(_gap)} repaired rasters absent from the bundle, e.g. {_gap[:3]}'
if LIMIT: tokens = tokens[:LIMIT]

In [ ]:
# E9: the temporal arm is capped at 640x360. At the default 1280x720 the 4-bit path
# emits degenerate all-'!' output from four images up (measured on the 3B, 2026-09-15) --
# silently, at a normal latency, so the whole run would produce nothing. Applied by
# CONDITION and never by image count: v4_both sends two images at full size and is
# already complete, and a count rule would retroactively change it.
_bekw = {'model': MODEL} if MODEL else {}
if CONDITION == 'v6_temporal':
    _bekw['max_image_px'] = V.MULTI_IMAGE_MAX_PX
    print('temporal arm: images capped at', V.MULTI_IMAGE_MAX_PX)
be = V.TransformersBackend(**_bekw)
print('loaded:', be.model_name)

# ONE call configuration, used by the smoke AND the run. The smoke used to build its own
# from the tag prompt, so under F5 it called build_prompt('f5_nuscenes_qa') and raised
# KeyError after the model had loaded (F-091). A smoke that does not make the run's call
# proves nothing about the run.
if F5:
    # One call per QUESTION, a one-word JSON answer, and tags=[] because the reply is not
    # tag-shaped. 64 new tokens is ample for {"answer": "..."} and deletes almost all of the
    # ~20 s spent decoding 35 booleans in the other arms.
    import src.nuqa as Q
    run_tokens = list(F5Q)          # question ids, not frame tokens
    RUN_KW = dict(prompt_for=lambda qid: Q.ANSWER_PROMPT.format(question=F5Q[qid]['question']),
                  tags=[], max_new_tokens=64)
else:
    # A SEQUENTIAL slice, and that is defensible here for one specific reason: E10 is a
    # PAIRED comparison. Both models see identical frames, so any bias in the slice hits both
    # arms and largely cancels in the delta, which is the quantity E10 reports. Measured on
    # tokens[:150]: all 35 tags have positives (none dead, none below 5), 24 of 35 clear the
    # 30-positive floor, and is_going_straight is 66.7% against the 628's 72.6%. It spans 30
    # scenes rather than 139, so it is NOT a basis for absolute per-tag claims about either
    # model -- only for the difference between them.
    run_tokens = tokens[:SLICE] if SLICE else tokens
    # n_frames is REQUIRED by PROMPT_V6_TEMPORAL's format string. Omitting it raised
    # KeyError only after the model had loaded - the guard cell passed while the real call
    # was broken, which is F-065's exact shape.
    RUN_KW = dict(text_for=text_for, tags=P.scoreable_tags(schema),
                  prompt_text=P.build_prompt(CONDITION, schema,
                      **({'n_frames': 5} if CONDITION == 'v6_temporal' else {})))

# SMOKE FIRST. The '!' failure is invisible in the progress line, and run_batch now aborts
# after 5 consecutive parse failures -- but on a 5.2 h job the cheap thing is to spend two
# minutes proving the configuration before committing the session (F-060).
_smoke = V.run_batch(be, run_tokens[:2], CONDITION, OUT/'_smoke.jsonl',
                     images_for=images_for, progress=False, **RUN_KW)
_rows = [json.loads(l) for l in (OUT/'_smoke.jsonl').open()]
assert all(r['parse_reason'].startswith('ok') for r in _rows), (
    f"smoke failed: {[r['parse_reason'] for r in _rows]}. If raw is all '!', the vision "
    "token count is too high -- lower max_image_px further before spending the session.")
print('smoke ok:', [(r['parse_reason'], r['n_answered']) for r in _rows])


In [ ]:
# ONE definition of the results filename, shared with scripts/preflight.py. When preflight
# computed it separately it checked the wrong file and called E10 blocked by rows that
# belong to a different model (R4: never write the same thing twice).
RESULT = OUT/V.result_filename(CONDITION, MODEL)
stats = V.run_batch(be, run_tokens, CONDITION, RESULT, images_for=images_for,
                    progress=True, **RUN_KW)
print(stats)


## Check, then take the results home

`n_parse_fail` is RQ2b. A high `missing:` count for one tag means the model renamed it —
the 3B did that to `stopped_vehicle` on 86% of frames, the 7B on none (F-057, F-059).

The file under `/kaggle/working/results/` is saved as notebook output when you commit.
Download it and put it in `outputs/results/` in the repo, then rebuild the bundle so the
next session resumes from it.

In [ ]:
import collections, statistics as st
rows = [json.loads(l) for l in RESULT.open()]
print(f'rows            : {len(rows)}')
print(f'parse ok        : {sum(r["parse_reason"].startswith("ok") for r in rows)}/{len(rows)}')
print(f'backend errors  : {sum(r["backend_error"] for r in rows)}')
_nt = len(RUN_KW['tags'])
print(f'answered {_nt}/{_nt}  : {sum(r["n_answered"]==_nt for r in rows)}')
print(f'median wall/cpu : {st.median(r["latency_s"] for r in rows):.1f}s / '
      f'{st.median(r["cpu_s"] for r in rows):.1f}s')
for p, c in collections.Counter(p for r in rows for p in r['problems']).most_common(10):
    print(f'   {c:5d}x  {p}')
print(f'\nresults at: {RESULT}')